<a href="https://colab.research.google.com/github/DOJO-Smart-Ways/DOJO-Beam-Transforms/blob/pbi-footprint/pbi_footprint/pbi_zcce100I_pipeline_to_staging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Configs


In [ ]:
!pip install git+https://github.com/DOJO-Smart-Ways/DOJO-Beam-Transforms.git@main#egg=dojo-beam-transforms

In [2]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from apache_beam.runners import DataflowRunner
from datetime import datetime, timedelta, date

import os
from google.cloud import bigquery

# Google Auth
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

# GCP Project
os.environ["GOOGLE_CLOUD_PROJECT"] = 'nidec-ga'

Authenticated


In [3]:
from pipeline_components import data_enrichment as de
from pipeline_components import data_cleaning as dc

# Pipeline Options

In [4]:
pipeline_options = {
  'project': 'nidec-ga' ,
  'runner': 'DataflowRunner',
  'region': 'us-central1',
  'staging_location': 'gs://nidec-ga-temp/data-flow-pipelines/pbi-footprint/staging',
  'temp_location': 'gs://nidec-ga-temp/data-flow-pipelines/pbi-footprint/temp',
  'template_location': 'gs://nidec-ga-temp/data-flow-pipelines/pbi-footprint/template/pbi-zcce100I',
  'sdk_container_image': 'us-central1-docker.pkg.dev/nidec-ga/dojo-beam/dojo_beam',
  'sdk_location': 'container'
}

# pipeline_options = {'project': 'nidec-ga'}

pipeline_options = PipelineOptions.from_dictionary(pipeline_options)
pipeline = beam.Pipeline(options=pipeline_options)

# Aux Methods

In [5]:
class DateTimeToString(beam.DoFn):
    def __init__(self, columns):
        # Initialize with the name of the column to update
        self.columns = columns

    def process(self, element):

        for column in self.columns:
          datetime_value = element.get(column,'')
          # Convert the datetime value to string in the specified format
          datetime_string = datetime_value.strftime('%Y-%m-%d %H:%M:%S')
          # Update the specified column with the datetime string
          element[column] = datetime_string

        # Output the modified element
        yield element

In [6]:
def get_output_schema() -> str:
    schema = """
      KALAID:STRING,
      KALADAT:DATETIME,
      BEWTP:STRING,
      BELNR:STRING,
      BUZEI:INTEGER,
      ZEKKN:INTEGER,
      EBELN:STRING,
      EBELP:INTEGER,
      LIFNR:STRING,
      BUDAT:DATETIME,
      MENGE:FLOAT,
      MEINS:STRING,
      BPMNG:FLOAT,
      BPRME:STRING,
      DMBTR:FLOAT,
      WRBTR:FLOAT,
      WAERS:STRING,
      WERKS:STRING,
      MATNR:STRING,
      XBLNR:STRING,
      LFGJA:INTEGER,
      BWART:STRING,
      BLART:STRING,
      ZPVAT:FLOAT,
      ZCOF:FLOAT,
      ZICMS:FLOAT,
      ZIPI:FLOAT,
      ZVAT:STRING,
      INCO1:STRING,
      ZPFRN:FLOAT,
      ZFRNC:FLOAT,
      ZFRNL:FLOAT,
      ZPFRI:FLOAT,
      ZPIIM:FLOAT,
      ZPDIM:FLOAT,
      ZFRIC:FLOAT,
      ZFRIL:FLOAT,
      ZIIMC:FLOAT,
      ZIIML:FLOAT,
      ZDIMC:FLOAT,
      ZDIML:FLOAT,
      ZBD1T:INTEGER,
      ZPJURO:FLOAT,
      ZJURO:FLOAT,
      UKURS:FLOAT,
      USNAM:STRING,
      CPUDT:DATETIME,
      CPUTM:DATETIME,
      EKORG:STRING,
      ALAND:STRING,
      ZPJURP:FLOAT,
      ZPJURF:FLOAT,
      MTART:STRING
    """

    return schema

# Pipeline builds

In [7]:
def run_pipeline(output_table, input_data_zcce100I, input_data_zcce100I_sheets, tab_zcce100I_sheets, temp_location):
  import pandas as pd

  input_zcce100I = (
    pipeline
    | 'Read from BigQuery' >> beam.io.ReadFromBigQuery(query='SELECT * FROM nidec-ga.bq_transient.ZCCE100I', use_standard_sql=True, gcs_location=temp_location)
  )

  input_data_zcce100I_sheets = (
      pipeline
      | 'Read From Storage Transient -- Loyalty' >> beam.Create([input_data_zcce100I_sheets])
      | 'Read Excel -- Loyalty' >> beam.FlatMap(lambda file: pd.read_excel(file, tab_zcce100I_sheets).to_dict('records'))
  )

  clean_zcce100I = (
      input_zcce100I
      | 'To Datetime' >> beam.ParDo(DateTimeToString(['KALADAT', 'BUDAT', 'CPUDT', 'CPUTM']))
  )


  group_zcce100I = de.key_transform(clean_zcce100I, ['KALAID'], 'ZCCE100I')

  group_zcce100I_sheets = de.key_transform(input_data_zcce100I_sheets, ['KALAID'], 'ZCCE100I Sheets')


  zcce100I = de.join(group_zcce100I, group_zcce100I_sheets, method='inner_join')


  zcce100I | 'Write To BigQuery' >> beam.io.WriteToBigQuery(
      output_table,
      schema=get_output_schema(),
      create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED,
      write_disposition=beam.io.BigQueryDisposition.WRITE_APPEND,
      custom_gcs_temp_location = temp_location
  )


  pipeline.run().wait_until_finish()

In [8]:
# Call Pipeline and pass parameters
if __name__ == '__main__':
    output_table = 'nidec-ga.bq_staging.ZCCE100I'
    input_data_zcce100I = 'nidec-ga.bq_transient.ZCCE100I'
    input_data_zcce100I_sheets = 'gs://nidec-ga-transient/pbi-supplier-footprint/PFX ZCCE100I Cockpits Selection.xlsx'
    tab_zcce100I_sheets = 'Cockpits'
    temp_location = 'gs://nidec-ga-temp/data-flow-pipelines/pbi-footprint/temp/'
    run_pipeline(output_table, input_data_zcce100I, input_data_zcce100I_sheets, tab_zcce100I_sheets, temp_location)